# Compound Name Mapping

Map BRD identifiers to proper compound names and upload to Snowflake table.

## Data Sources
- PubChem: Retrieve common names and IUPAC names from InChIKey
- ChEMBL: Supplement with drug names

## Processing Time
- Approximately 5,000 compounds x 0.6 sec/compound = 50 minutes

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import logging

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Project root
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

# Module imports
from compound_name_mapping import CompoundNameMapper, upload_to_snowflake

# Results directory
results_dir = project_root / 'results' / 'compound_name_mapping'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Results directory: {results_dir}')

## 1. Snowflake Connection

In [ ]:
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect

def load_private_key():
    key_path = os.path.expanduser('~/.ssh/snowflake_rsa_key.pem')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(), password=None, backend=default_backend()
        )
    return private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )

conn = connect(
    user="KOREEDA",
    account="DUETMBM-LL33279",
    private_key=load_private_key(),
    warehouse="BIOINFORMATICS_XS",
    database="BIOINFORMATICS",
    schema="LINCS",
    role="ACCOUNTADMIN",
)
print('Snowflake connection successful')

## 2. Retrieve Unique Compound List

In [ ]:
query = """
SELECT DISTINCT
    "pertname",
    "pertid",
    "inchi_key",
    "canonical_smiles"
FROM BIOINFORMATICS.LINCS.GLYCO_GENES_WIDE
WHERE "inchi_key" IS NOT NULL
ORDER BY "pertname"
"""

df_compounds = pd.read_sql(query, conn)
print(f'Unique compounds: {len(df_compounds):,}')
print(f'\nBRD compounds: {df_compounds["pertname"].str.startswith("BRD-").sum():,}')
df_compounds.head()

## 3. Execute Compound Name Mapping

**Note**: This takes approximately 50 minutes. Re-runs will be faster if cache exists.

In [ ]:
# Initialize mapper
mapper = CompoundNameMapper(cache_dir=results_dir, rate_limit=0.3)

# Execute processing
print('Starting compound name mapping...')
df_mapping = mapper.process_compounds(df_compounds, save_interval=100)

print(f'\nProcessing complete: {len(df_mapping):,} records')
print(f'\nRecords by name source:')
print(df_mapping['name_source'].value_counts())

## 4. Review Results

In [ ]:
# BRD compound name mapping results
df_brd = df_mapping[df_mapping['pertname'].str.startswith('BRD-')]

print(f'BRD compounds: {len(df_brd)} records')
print(f'\nSuccessfully mapped (non-BRD names): {(df_brd["compound_name"] != df_brd["pertname"]).sum()} records')

print('\n=== BRD Compound Mapping Examples (Top 20) ===')
for _, row in df_brd.head(20).iterrows():
    if row['compound_name'] != row['pertname']:
        print(f"{row['pertname']} -> {row['compound_name']}")

In [ ]:
# Save locally
mapper.save_to_parquet(df_mapping)
df_mapping.to_csv(results_dir / 'compound_mapping.csv', index=False)
print(f'Save complete: {results_dir}')

## 5. Upload to Snowflake

In [ ]:
# Upload to Snowflake
success = upload_to_snowflake(df_mapping, conn, table_name='COMPOUND_NAME_MAPPING')

if success:
    print('\nSnowflake upload successful!')
    print('Table: BIOINFORMATICS.LINCS.COMPOUND_NAME_MAPPING')

In [ ]:
# Verification query
verify_query = """
SELECT PERTNAME, COMPOUND_NAME, NAME_SOURCE, CHEMBL_NAME, PUBCHEM_NAME
FROM BIOINFORMATICS.LINCS.COMPOUND_NAME_MAPPING
WHERE PERTNAME LIKE 'BRD-%'
AND COMPOUND_NAME != PERTNAME
LIMIT 20
"""

df_verify = pd.read_sql(verify_query, conn)
print('=== Verification from Snowflake ===')
df_verify

In [ ]:
# Close connection
conn.close()
print('\nComplete!')